In [8]:
# imports

In [9]:
from pathlib import Path
import json
import pandas as pd

In [6]:
# test the file discovery

In [2]:
from pathlib import Path

folder_path = Path(r"C:\Users\stanl\Project 30\Datasets")

json_files = list(folder_path.rglob("*.json"))

print("Number of JSON files found:", len(json_files))
print("First 5 files:")
for f in json_files[:5]:
    print(f)

Number of JSON files found: 75000
First 5 files:
C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_1.json
C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_10.json
C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_100.json
C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_1000.json
C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_101.json


In [5]:
# test opening one JSON file

In [3]:
import json
folder_path = Path(r"C:\Users\stanl\Project 30\Datasets")
json_files = list(folder_path.rglob("*.json"))

sample_file = json_files[0]
print("Sample file:", sample_file)

with open(sample_file, "r", encoding="utf-8") as f:
    article = json.load(f)

print(type(article))
print(article.keys())

Sample file: C:\Users\stanl\Project 30\Datasets\Financial and Economic News_positive_20260322072615\Financial and Economic News_positive_20260322072615\article_1.json
<class 'dict'>
dict_keys(['thread', 'uuid', 'url', 'ord_in_thread', 'author', 'published', 'title', 'text', 'summary', 'highlightText', 'highlightTitle', 'highlightThreadTitle', 'language', 'sentiment', 'categories', 'topics', 'ai_allow', 'has_canonical', 'breaking', 'webz_reporter', 'external_links', 'external_images', 'internal_images', 'entities', 'syndication', 'trust', 'rating', 'crawled', 'updated'])


In [7]:
import pandas as pd

folder_path = Path(r"C:\Users\stanl\Project 30\Datasets")

all_rows = []

for file_path in folder_path.rglob("*.json"):
    with open(file_path, "r", encoding="utf-8") as f:
        article = json.load(f)

    thread = article.get("thread", {})
    entities = article.get("entities", {})

    row = {
        "article_id": article.get("uuid"),
        "title": article.get("title"),
        "text": article.get("text"),
        "published_at": article.get("published"),
        "url": article.get("url"),
        "author": article.get("author"),
        "language": article.get("language"),
        "source_site": thread.get("site"),
        "source_country": thread.get("country"),
        "categories_raw": str(article.get("categories")),
        "entity_persons_raw": str(entities.get("persons")),
        "entity_organizations_raw": str(entities.get("organizations")),
        "entity_locations_raw": str(entities.get("locations")),
        "has_entity_metadata": int(
            bool(entities.get("persons") or entities.get("organizations") or entities.get("locations"))
        ),
        "raw_file_name": file_path.name
    }

    all_rows.append(row)

df = pd.DataFrame(all_rows)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print(df.head())

df.to_csv("raw_ingested_articles.csv", index=False)
print("Saved raw_ingested_articles.csv")

Rows: 75000
Columns: ['article_id', 'title', 'text', 'published_at', 'url', 'author', 'language', 'source_site', 'source_country', 'categories_raw', 'entity_persons_raw', 'entity_organizations_raw', 'entity_locations_raw', 'has_entity_metadata', 'raw_file_name']
                                 article_id  \
0  71e3502a665cb219fbf1180ffa8e3d83e48d0a5f   
1  7e1f5b18780831791cf76ed3ed83620a5dd299dd   
2  3ba4c8cebb884f3c37afb91d82b3c220b6a16c51   
3  8bf04d9e8e0e8ff7ed5ccd40da0f269bdae3af7f   
4  55b3f91372c0eed8acd2759be6724bd4c273f84a   

                                               title  \
0  Philippine Peso Hits Record Lows: Is Now the B...   
1  Power Assets Earnings: Eyes on Use of UKPN Sal...   
2  Oil rises after Iran strikes Middle East energ...   
3  The United States and Japan have agreed to inv...   
4  AIA Group Ltd Has $9.68 Million Position in Th...   

                                                text  \
0  Dubai: Filipino expats in the UAE are seeing s...   
1  Po

In [11]:
df = pd.read_csv("raw_ingested_articles.csv")

# fill missing values
df["title"] = df["title"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# remove weak rows
df = df[(df["title"].str.len() >= 10) & (df["text"].str.len() >= 100)]

# remove duplicates
df = df.drop_duplicates(subset=["article_id"])

print("Cleaned rows:", len(df))
print(df.head())

df.to_csv("raw_ingested_articles_cleaned.csv", index=False)
print("Saved raw_ingested_articles_cleaned.csv")

Cleaned rows: 62203
                                 article_id  \
0  71e3502a665cb219fbf1180ffa8e3d83e48d0a5f   
1  7e1f5b18780831791cf76ed3ed83620a5dd299dd   
2  3ba4c8cebb884f3c37afb91d82b3c220b6a16c51   
3  8bf04d9e8e0e8ff7ed5ccd40da0f269bdae3af7f   
4  55b3f91372c0eed8acd2759be6724bd4c273f84a   

                                               title  \
0  Philippine Peso Hits Record Lows: Is Now the B...   
1  Power Assets Earnings: Eyes on Use of UKPN Sal...   
2  Oil rises after Iran strikes Middle East energ...   
3  The United States and Japan have agreed to inv...   
4  AIA Group Ltd Has $9.68 Million Position in Th...   

                                                text  \
0  Dubai: Filipino expats in the UAE are seeing s...   
1  Power Assets Holdings Ltd\n00006: XHKG (HKG)\n...   
2  BEIJING: Oil prices rose on Thursday, with ben...   
3  The United States and Japan have agreed to inv...   
4  AIA Group Ltd Has $9.68 Million Position in Th...   

                    pub